# Banco de pruebas del HITL (human-in-the-loop)

Este notebook ejercita **los 4 puntos HITL** del meta-supervisor y **las 3 APIs** para atenderlos. Cada escenario es autocontenido: puedes correrlos en orden o saltar al que te interese.

## Mapa de los puntos HITL

| # | Punto | ¿Cuándo salta? | Decisiones que acepta |
|---|-------|----------------|------------------------|
| 1 | `revisar_plan` | **Siempre**, salvo `plan=` fijado | `{"accion": "aprobar"}` · `{"accion": "editar", "plan": {...}}` · `{"accion": "cancelar"}` |
| 2 | `tras_preguntas` | Opt-in: `pausas=["tras_preguntas"]` | `{}` (seguir) · `{"vetar": [<índice o texto exacto>, ...]}` — veta preguntas del examen y re-congela |
| 3 | `tras_diagnostico` | Opt-in: `pausas=["tras_diagnostico"]` | `{}` (seguir) · `{"restricciones": {"<banco>": {"fijar"/"familias"/"etiquetas": ...}}}` |
| 4 | `antes_fase_b` | Opt-in: `pausas=["antes_fase_b"]` | `{}` (seguir) · `{"saltar": True}` · `{"vetar": ["<etiqueta>", ...]}` |
| 5 | `tras_fase_b` | Opt-in **y solo si la Fase B corona** | `{}` (aprobar y persistir) · `{"vetar": True}` (descartar coronación, no persiste) |

## Cómo funciona por debajo

- Cada punto es un `interrupt()` de LangGraph. El grafo se **detiene**, devuelve un *payload* (lo que el humano debe ver) y espera un `Command(resume=<decisión>)`.
- El estado pausado vive en un **checkpointer en memoria** (`MemorySaver`) indexado por `thread_id`. Consecuencia: **la pausa y la reanudación deben ocurrir en el mismo proceso** — si reinicias el kernel entre `iniciar` y `reanudar`, el hilo se pierde.
- Los `interrupt()` van **al principio del nodo**: al reanudar, LangGraph re-ejecuta el nodo desde el inicio (por eso los prints del diagnóstico pueden aparecer dos veces; es normal, no un bug).

## Avisos antes de empezar

1. **Coste**: cada escenario ejecuta experimentos reales (embeddings + LLM juez). Usa un PDF pequeño y los planes mínimos de este notebook (1 banco, 1 run, parrilla restringida).
2. **Deduplicación**: el mismo PDF ya estudiado se omite. Todos los escenarios usan `forzar=True`.
3. `revisar_plan` **no salta** con `plan=` fijado (por diseño): así el escenario 5 prueba las pausas opt-in de forma controlada, sin ruido del punto 1.
4. `reanudar()` **auto-aprueba** cualquier pausa posterior a la primera. Para decidir en TODAS las pausas usa el callback `revisor` (escenario 5) o el control a bajo nivel (escenario 6).

---
## 0 · Preparación

Configura la ruta del PDF (pequeño: 5-15 páginas es ideal) y construye el supervisor. El `revisor espía` es la pieza central del notebook: registra **todos** los payloads en `PAYLOADS` para inspeccionarlos después, y decide según el punto.

In [ ]:
from pathlib import Path

from rag_lab import CONFIG, MetaSupervisor
from rag_lab.supervisor import Plan

# --- AJUSTA ESTO ---
PDF = "documento_pequeno.pdf"        # PDF barato para pruebas
assert Path(PDF).exists(), f"No encuentro {PDF}"

avisos = CONFIG.validar()
if avisos:
    print("Avisos de configuración:", *avisos, sep="\n  - ")

meta = MetaSupervisor()

# Registro global de payloads: aquí queda TODO lo que el grafo enseñó al humano.
PAYLOADS: list[dict] = []


def espiar(payload: dict) -> None:
    PAYLOADS.append(payload)
    punto = payload.get("punto", "revisar_plan")
    print(f"\n{'='*60}\n[HITL] Pausa en: {punto}")
    print(f"[HITL] Mensaje: {payload.get('mensaje', '(sin mensaje)')}")
    print(f"[HITL] Claves del payload: {sorted(payload)}")
    print(f"[HITL] Cómo responder: {payload.get('como_responder')}\n{'='*60}")

Un plan **mínimo y barato** que reutilizaremos: un solo banco (splitter), un run, y la parrilla restringida a dos etiquetas para que cada escenario tarde poco. Restricción y pausas son justo las palancas que el HITL permite tocar, así que además sirven de referencia de sintaxis.

In [ ]:
def plan_minimo(pausas: list[str] | None = None) -> dict:
    """Plan barato para pruebas: 1 banco, 1 run, parrilla de 2 configs."""
    return {
        "bancos": ["splitter"],
        "n_runs": 1,
        "presupuesto": {"max_runs": 1, "max_preguntas": 10},
        "restricciones": {"splitter": {"etiquetas": ["recursive_500", "recursive_1000"]}},
        "pausas": pausas or [],
    }

plan_minimo()

---
## 1 · `revisar_plan` con revisor — APROBAR

El caso base: el supervisor diagnostica, propone un plan y se detiene. Nuestro revisor lo mira (queda en `PAYLOADS`) y aprueba. Sin `plan=` fijado, este punto salta **siempre**.

In [ ]:
def revisor_aprobar(payload: dict) -> dict:
    espiar(payload)
    return {"accion": "aprobar"}

stack = meta.ejecutar(PDF, instruccion="prueba solo el splitter, un run, rapido",
                      revisor=revisor_aprobar, forzar=True)
print("\nStack final:", {k: v for k, v in stack.items() if not k.startswith("_")})

In [ ]:
# Inspección del payload que vio el humano: diagnóstico + plan propuesto.
p = PAYLOADS[-1]
print("Plan propuesto por el agente:")
print(p["plan_propuesto"])
print("\nPriors del diagnóstico (por banco):")
for banco, prior in p["diagnostico"]["bancos"].items():
    print(f"  {banco:10}:", "sin vecinos" if not prior.get("disponible")
          else f"típico={prior.get('ganador')} (acuerdo={prior.get('acuerdo')})")

---
## 2 · `revisar_plan` con revisor — EDITAR

El humano **corrige el plan del agente** antes de gastar un céntimo. El plan editado pasa por `normalizar_plan`, así que puedes mandarlo incompleto: bancos inválidos se filtran, los que falten se rellenan con defaults, y `n_runs` se satura a [1, 10]. Fíjate en el print `[meta] Plan editado por el usuario.`

In [ ]:
def revisor_editar(payload: dict) -> dict:
    espiar(payload)
    propuesto = payload["plan_propuesto"]
    print(f"[revisor] El agente proponía bancos={propuesto['bancos']}, "
          f"n_runs={propuesto['n_runs']}. Lo recorto al plan mínimo.")
    return {"accion": "editar", "plan": plan_minimo()}

stack = meta.ejecutar(PDF, revisor=revisor_editar, forzar=True)
print("\nStack final:", {k: v for k, v in stack.items() if not k.startswith("_")})

---
## 3 · `revisar_plan` con revisor — CANCELAR

Cancelar deja el plan sin bancos y el grafo termina por la ruta `END` **sin ejecutar nada** (ni experimentos ni escrituras en memoria). El stack devuelto debe salir vacío.

In [ ]:
def revisor_cancelar(payload: dict) -> dict:
    espiar(payload)
    return {"accion": "cancelar"}

stack = meta.ejecutar(PDF, revisor=revisor_cancelar, forzar=True)
assert not stack, f"Cancelar debería devolver un stack vacío y devolvió: {stack}"
print("OK: cancelación limpia, stack vacío.")

---
## 4 · `iniciar` / `reanudar` — el flujo manual de notebook

La API pensada para decidir **tú**, con calma, entre dos celdas. `iniciar` corre hasta `revisar_plan` y te devuelve `(thread_id, payload)`; examinas el payload todo el tiempo que quieras; `reanudar` continúa con tu decisión.

⚠️ Recuerda: el `thread_id` vive en memoria del proceso. No reinicies el kernel entre las dos celdas. Y `reanudar` auto-aprueba cualquier pausa opt-in posterior — para controlarlas todas, escenarios 5 y 6.

In [ ]:
thread_id, payload = meta.iniciar(PDF, instruccion="solo splitter, un run")
print("thread_id:", thread_id)
print("\nPlan propuesto:", payload["plan_propuesto"] if payload else "(no hubo pausa)")

In [ ]:
# Tómate tu tiempo aquí: mira payload["diagnostico"], decide...
# ...y cuando quieras, reanuda. Probamos la edición por esta vía:
stack = meta.reanudar(thread_id, {"accion": "editar", "plan": plan_minimo()})
print("\nStack final:", {k: v for k, v in stack.items() if not k.startswith("_")})

---
## 5 · Las tres pausas opt-in, con plan fijado

La combinación de precisión quirúrgica: `plan=` fijado **suprime** `revisar_plan` (punto 1) y las `pausas` declaradas activan los puntos 2-4. Un solo revisor **enruta por `payload["punto"]`** y toma una decisión distinta en cada parada:

- `tras_diagnostico` → inyecta una restricción (recorta la parrilla del splitter a una sola etiqueta).
- `antes_fase_b` → de momento, seguir (`{}`); abajo hay variantes para vetar o saltar.
- `tras_fase_b` → aprueba la coronación.

**Nota honesta sobre `tras_fase_b`**: solo salta si la Fase B llegó a coronar, y la Fase B solo corre si la Fase A dejó **empates reales** (≥2 finalistas en algún banco). Con el plan mínimo de 1 run y parrilla recortada puede que no haya empate y el punto 4 no se dispare — no es un fallo del HITL. Para maximizar la probabilidad de verlo: 2+ bancos, `n_runs=2`, sin restricciones (más caro).

In [ ]:
def revisor_por_punto(payload: dict) -> dict:
    espiar(payload)
    punto = payload.get("punto", "revisar_plan")

    if punto == "tras_preguntas":
        print(f"[revisor] Examen: {len(payload['preguntas'])} preguntas, "
              f"tasa de verificación={payload.get('tasa_verificacion')}, "
              f"congelado reutilizado={payload.get('congelado_reutilizado')}")
        for q in payload["preguntas"][:3]:
            print(f"  [{q['i']}] {q['question'][:70]}")
        return {}          # o {"vetar": [0]} para vetar la pregunta 0

    if punto == "tras_diagnostico":
        print("[revisor] Inyecto restricción: splitter solo 'recursive_500'.")
        return {"restricciones": {"splitter": {"etiquetas": ["recursive_500",
                                                             "recursive_1000"]}}}
    if punto == "antes_fase_b":
        print("[revisor] Stack sintetizado:", payload.get("stack"))
        print("[revisor] Pendiente de Fase B:", payload.get("pendiente_fase_b"))
        return {}                       # seguir; variantes más abajo

    if punto == "tras_fase_b":
        print("[revisor] Campeón end-to-end:", payload.get("campeon"))
        return {}                       # aprobar => se persiste en memoria

    return {"accion": "aprobar"}        # por si acaso


plan_con_pausas = {
    "bancos": ["splitter", "embedding"],   # 2 bancos para dar opciones a la Fase B
    "n_runs": 2,
    "presupuesto": {"max_runs": 2, "max_preguntas": 10},
    "pausas": ["tras_preguntas", "tras_diagnostico", "antes_fase_b", "tras_fase_b"],
}

stack = meta.ejecutar(PDF, plan=plan_con_pausas, revisor=revisor_por_punto, forzar=True)
print("\nStack final:", {k: v for k, v in stack.items() if not k.startswith("_")})
print("\nPuntos por los que pasó esta corrida:",
      [p.get("punto", "revisar_plan") for p in PAYLOADS[-3:]])

### Variantes de `antes_fase_b`: vetar y saltar

Dos revisores alternativos para el mismo plan. El **veto por etiqueta** elimina finalistas concretos del cruce factorial (verás `[meta] Candidatos vetados en la pausa: [...]`); el **salto** omite la Fase B entera y el stack sale marcado con `fase_b: {"omitida_por_usuario": True}`.

In [ ]:
def revisor_vetando(payload: dict) -> dict:
    espiar(payload)
    if payload.get("punto") == "antes_fase_b":
        pendientes = payload.get("pendiente_fase_b") or {}
        # Vetamos el primer finalista NO líder que encontremos (demo).
        for banco, etiquetas in pendientes.items():
            if len(etiquetas) > 1:
                print(f"[revisor] Veto '{etiquetas[-1]}' del banco {banco}.")
                return {"vetar": [etiquetas[-1]]}
        print("[revisor] No hay nada que vetar (sin empates); sigo.")
    return {}

stack = meta.ejecutar(PDF, plan=plan_con_pausas, revisor=revisor_vetando, forzar=True)

In [ ]:
def revisor_saltando(payload: dict) -> dict:
    espiar(payload)
    if payload.get("punto") == "antes_fase_b":
        print("[revisor] Salto la Fase B: me quedo con la síntesis de la Fase A.")
        return {"saltar": True}
    return {}

stack = meta.ejecutar(PDF, plan=plan_con_pausas, revisor=revisor_saltando, forzar=True)
print("\nMarca de omisión:", stack.get("fase_b"))
assert stack.get("fase_b", {}).get("omitida_por_usuario"), "debería estar marcada la omisión" 

### Variante de `tras_fase_b`: VETAR la coronación

El veto en el punto 4 descarta el campeón de la Fase B **antes de persistir**: la memoria (y los priors futuros) no se contaminan con una coronación que el humano rechazó. Verás `[meta] Coronacion de Fase B vetada por el usuario: no se persiste.` — si la Fase B llegó a coronar, claro.

In [ ]:
def revisor_veta_coronacion(payload: dict) -> dict:
    espiar(payload)
    if payload.get("punto") == "tras_fase_b":
        print("[revisor] VETO la coronación end-to-end.")
        return {"vetar": True}
    return {}

stack = meta.ejecutar(PDF, plan=plan_con_pausas, revisor=revisor_veta_coronacion, forzar=True)

---
## 6 · Control a bajo nivel: pausa a pausa con `Command(resume=...)`

`ejecutar(revisor=...)` decide con un callback y `reanudar` auto-aprueba lo que venga después. Si quieres el control **absolutamente manual** de cada pausa (por ejemplo, para construir tu propia interfaz), habla con el grafo directamente. Es exactamente lo que `ejecutar` hace por dentro, pero deteniéndote tú en cada `__interrupt__`.

De regalo: `meta.grafo.get_state(config)` te enseña el estado del checkpoint entre pausas (qué nodo sigue, qué hay en el estado).

In [ ]:
import uuid as _uuid
from langgraph.types import Command

thread = str(_uuid.uuid4())
config = {"configurable": {"thread_id": thread}}

entrada = {"pdf_path": PDF, "instruccion": "", "plan_fijado": True,
           "plan": plan_con_pausas}   # requiere forzar manualmente: borra antes la
                                       # entrada de memoria o usa un PDF nuevo si
                                       # el documento ya consta como estudiado

estado = meta.grafo.invoke(entrada, config)

n_pausa = 0
while estado.get("__interrupt__"):
    n_pausa += 1
    payload = estado["__interrupt__"][0].value
    punto = payload.get("punto", "revisar_plan")
    print(f"\n>>> PAUSA {n_pausa}: {punto}")
    print(">>> siguiente nodo según el checkpoint:", meta.grafo.get_state(config).next)

    # AQUÍ decides tú, pausa a pausa. Demo: decisiones distintas por punto.
    if punto == "tras_preguntas":
        decision = {"seguir": True}   # o {"vetar": [0]}; nunca {} a pelo (falsy)
    elif punto == "tras_diagnostico":
        decision = {"seguir": True}
    elif punto == "antes_fase_b":
        decision = {"seguir": True}
    elif punto == "tras_fase_b":
        decision = {"seguir": True}
    else:
        decision = {"accion": "aprobar"}
    print(">>> respondo:", decision)

    estado = meta.grafo.invoke(Command(resume=decision), config)

print(f"\nTerminado tras {n_pausa} pausas.")
print("Stack final:", {k: v for k, v in estado.get("stack_final", {}).items()
                       if not k.startswith("_")})

**Detalle a observar**: `ejecutar` con `plan=` hace la comprobación de deduplicación *antes* de invocar el grafo; al invocar el grafo directamente esa comprobación **no existe**, así que este escenario reprocesa siempre. Es otra diferencia práctica entre las dos APIs.

---
## 7 · Revisor humano de verdad: `input()`

Para sentir el HITL como usuario final: un revisor que te pregunta por consola en cada pausa. Escribe `a` (aprobar/seguir), `s` (saltar Fase B), `v` (vetar) o `c` (cancelar el plan).

In [ ]:
def revisor_humano(payload: dict) -> dict:
    espiar(payload)
    punto = payload.get("punto", "revisar_plan")
    r = input(f"[{punto}] a=aprobar/seguir | s=saltar | v=vetar | c=cancelar > ").strip().lower()

    if punto == "revisar_plan":
        return {"accion": "cancelar"} if r == "c" else {"accion": "aprobar"}
    if punto == "antes_fase_b":
        if r == "s":
            return {"saltar": True}
        if r == "v":
            ets = input("  etiquetas a vetar (coma): ").split(",")
            return {"vetar": [e.strip() for e in ets if e.strip()]}
        return {}
    if punto == "tras_fase_b":
        return {"vetar": True} if r == "v" else {}
    return {}

# Descomenta para probarlo en vivo:
# stack = meta.ejecutar(PDF, plan=plan_con_pausas, revisor=revisor_humano, forzar=True)

---
## 8 · Autopsia y checklist

Todo lo que el grafo enseñó al humano quedó en `PAYLOADS`. Repásalo y verifica la cobertura de los 4 puntos.

In [ ]:
from collections import Counter

vistos = Counter(p.get("punto", "revisar_plan") for p in PAYLOADS)
esperados = ["revisar_plan", "tras_preguntas", "tras_diagnostico", "antes_fase_b", "tras_fase_b"]

print(f"Payloads capturados: {len(PAYLOADS)}\n")
for punto in esperados:
    marca = "✓" if vistos.get(punto) else "✗"
    nota = ""
    if punto == "tras_fase_b" and not vistos.get(punto):
        nota = "  (normal si la Fase B no llegó a coronar: sin empates reales)"
    print(f"  {marca} {punto:18} visto {vistos.get(punto, 0)} veces{nota}")

### Chuleta final de esquemas de decisión

```python
# revisar_plan (siempre, salvo plan= fijado)
{"accion": "aprobar"}
{"accion": "editar", "plan": {"bancos": [...], "n_runs": 2, ...}}   # pasa por normalizar_plan
{"accion": "cancelar"}                                               # plan sin bancos -> END

# tras_preguntas (opt-in) — examen de la sesión, verificado y congelado por doc_hash
{}                                                                   # seguir
{"vetar": [0, 2]}                                                    # vetar por índice...
{"vetar": ["¿Texto exacto de la pregunta?"]}                         # ...o por texto; re-congela el examen

# tras_diagnostico (opt-in)
{}                                                                   # seguir
{"restricciones": {"splitter":  {"fijar": {"name": "recursive", "chunk_size": 500, "chunk_overlap": 50}},
                   "retriever": {"familias": ["bm25", "ensemble"]},
                   "embedding": {"etiquetas": ["bge-m3"]}}}          # se FUSIONAN con las del plan

# antes_fase_b (opt-in)
{}                                                                   # seguir al cruce factorial
{"saltar": True}                                                     # stack["fase_b"]={"omitida_por_usuario": True}
{"vetar": ["<etiqueta>", ...]}                                       # elimina finalistas del factorial

# tras_fase_b (opt-in; solo si la Fase B coronó)
{}                                                                   # aprobar => registrar_decision (persiste)
{"vetar": True}                                                      # descartar coronación, NO persiste
```

### Recordatorios

- `MemorySaver` es volátil: pausa y reanudación **en el mismo proceso/kernel**.
- Los `interrupt()` están al principio del nodo → el nodo se re-ejecuta al reanudar (prints duplicados = normal).
- `plan=` fijado suprime `revisar_plan` pero **no** las pausas declaradas en `pausas`.
- `reanudar()` auto-aprueba las pausas posteriores; para controlarlas todas: `revisor` o el bucle del escenario 6.
- `forzar=True` mientras pruebes, o la deduplicación te devolverá el stack antiguo sin ejecutar.
- Tras estas pruebas, tu memoria (`rag_memory/memoria_unificada.json`) contiene corridas de juguete: bórrala antes de las corridas serias, como ya tenías previsto.